In [21]:
import os
import numpy as np
import matplotlib.pyplot as plt

import h5py

# import h5 reader
from SciFiReaders import NSIDReader #AutoReader

In [22]:
path = '/Users/7c2/Desktop/2026_09_30_tp4_firsttry/'

files = os.listdir(path)
files.sort()
files

['austin_test_ep0_ch1_proc0.h5',
 'austin_test_ep0_ch1_proc1.h5',
 'austin_test_ep0_ch1_proc10.h5',
 'austin_test_ep0_ch1_proc11.h5',
 'austin_test_ep0_ch1_proc12.h5',
 'austin_test_ep0_ch1_proc13.h5',
 'austin_test_ep0_ch1_proc14.h5',
 'austin_test_ep0_ch1_proc15.h5',
 'austin_test_ep0_ch1_proc2.h5',
 'austin_test_ep0_ch1_proc3.h5',
 'austin_test_ep0_ch1_proc4.h5',
 'austin_test_ep0_ch1_proc5.h5',
 'austin_test_ep0_ch1_proc6.h5',
 'austin_test_ep0_ch1_proc7.h5',
 'austin_test_ep0_ch1_proc8.h5',
 'austin_test_ep0_ch1_proc9.h5',
 'austin_test_ep1_ch1_proc0.h5',
 'austin_test_ep1_ch1_proc1.h5',
 'austin_test_ep1_ch1_proc10.h5',
 'austin_test_ep1_ch1_proc11.h5',
 'austin_test_ep1_ch1_proc12.h5',
 'austin_test_ep1_ch1_proc13.h5',
 'austin_test_ep1_ch1_proc14.h5',
 'austin_test_ep1_ch1_proc15.h5',
 'austin_test_ep1_ch1_proc2.h5',
 'austin_test_ep1_ch1_proc3.h5',
 'austin_test_ep1_ch1_proc4.h5',
 'austin_test_ep1_ch1_proc5.h5',
 'austin_test_ep1_ch1_proc6.h5',
 'austin_test_ep1_ch1_proc7.h5'

In [23]:
# Open the same file directly with h5py and inspect its contents.
# If this file is locked by another reader, close that reader first,
# or choose another file (for example, files[1]).
h5_path = os.path.join(path, files[0])

with h5py.File(h5_path, "r") as h5:
    print("File:", h5_path)
    print("File attributes:", dict(h5.attrs))

    def show_dataset(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"{name}: shape={obj.shape}, dtype={obj.dtype}")

    h5.visititems(show_dataset)  # Also finds datasets inside nested groups.


File: /Users/7c2/Desktop/2026_09_30_tp4_firsttry/austin_test_ep0_ch1_proc0.h5
File attributes: {}
control: shape=(3,), dtype=uint64
ftoa_fall: shape=(3410614,), dtype=uint8
ftoa_rise: shape=(3410614,), dtype=uint8
lvds_timestamp: shape=(0,), dtype=uint64
pileup: shape=(3410614,), dtype=uint8
sw_lvds_fe_timestamp: shape=(0,), dtype=uint64
sw_lvds_re_timestamp: shape=(0,), dtype=uint64
sw_pulse_timestamp: shape=(0,), dtype=uint64
sw_ttl_fe_timestamp: shape=(0,), dtype=uint64
sw_ttl_re_timestamp: shape=(0,), dtype=uint64
toa: shape=(3410614,), dtype=uint64
tot: shape=(3410614,), dtype=uint16
ttl_timestamp: shape=(0,), dtype=uint64
uftoa_start: shape=(3410614,), dtype=uint8
uftoa_stop: shape=(3410614,), dtype=uint8
x: shape=(3410614,), dtype=uint16
y: shape=(3410614,), dtype=uint16


In [24]:
# Read just the first 10 events: slicing avoids loading the whole file.
# These dataset names are present in your H5 files.
with h5py.File(h5_path, "r") as h5:
    x_sample = h5["x"][:10]
    y_sample = h5["y"][:10]
    toa_sample = h5["toa"][:10]
    tot_sample = h5["tot"][:10]
    print("toa attributes:", dict(h5["toa"].attrs))

# The slices are NumPy arrays, so they remain usable after the file closes.
print("x:", x_sample)
print("y:", y_sample)
print("toa:", toa_sample)
print("tot:", tot_sample)


toa attributes: {}
x: [ 51  77 150 149 251 229 302 202 308 128]
y: [ 72 165 140 120 189 135 166 115 109 162]
toa: [93995061 93995017 93995050 93995098 93995028 93995083 93995074 93995134
 93995050 93995020]
tot: [12  2  5  7 20 23  6  3 15  9]


### Lazy event table and memory-managed time sort
Each partition reads at most 100,000 events. Column dtypes are preserved; `file_id` and `event_id` identify the original event. `fields=None` includes every root-level 1D column aligned with `toa`; empty control/timestamp streams are excluded and remain in the source H5 files.

Only shapes and dtypes are read during construction. Close any existing NSIDReader first (`reader.close()`) if it locks an input. Do not use `events.compute()` or `events.persist()` for the full collection.

Before combining epochs, confirm that their `toa` values share the same clock origin and units. Sorting raw timestamps does not correct clock resets or rollover.


In [26]:
from event_pipeline import build_events, sort_to_parquet
import dask.dataframe as dd

events, manifest = build_events(path, batch_rows=100_000, fields=None)
print(events.dtypes)
print("Partitions:", events.npartitions)
manifest  # Maps file_id to source filename and event count.


ftoa_fall       uint8
ftoa_rise       uint8
pileup          uint8
toa            uint64
tot            uint16
uftoa_start     uint8
uftoa_stop      uint8
x              uint16
y              uint16
file_id        uint32
event_id       uint64
dtype: object
Partitions: 874


,file_id,filename,events
0,0,/Users/7c2/Desktop/2026_09_30_tp4_firsttry/aus...,3410614
1,1,/Users/7c2/Desktop/2026_09_30_tp4_firsttry/aus...,3416082
2,2,/Users/7c2/Desktop/2026_09_30_tp4_firsttry/aus...,3389125
3,3,/Users/7c2/Desktop/2026_09_30_tp4_firsttry/aus...,3385894
4,4,/Users/7c2/Desktop/2026_09_30_tp4_firsttry/aus...,3388539
5,5,/Users/7c2/Desktop/2026_09_30_tp4_firsttry/aus...,3400531
6,6,/Users/7c2/Desktop/2026_09_30_tp4_firsttry/aus...,3414246
7,7,/Users/7c2/Desktop/2026_09_30_tp4_firsttry/aus...,3409351
8,8,/Users/7c2/Desktop/2026_09_30_tp4_firsttry/aus...,3412362
9,9,/Users/7c2/Desktop/2026_09_30_tp4_firsttry/aus...,3412874


In [27]:
# Read only the first partition's first 10 rows.
events.head(10, npartitions=1)


,ftoa_fall,ftoa_rise,pileup,toa,tot,uftoa_start,uftoa_stop,x,y,file_id,event_id
0,1,7,0,93995061,12,0,6,51,72,0,0
1,5,6,0,93995017,2,7,1,77,165,0,1
2,13,7,0,93995050,5,0,2,150,140,0,2
3,11,15,0,93995098,7,0,7,149,120,0,3
4,9,4,0,93995028,20,0,2,251,189,0,4
5,15,1,0,93995083,23,0,1,229,135,0,5
6,14,4,0,93995074,6,0,2,302,166,0,6
7,3,12,0,93995134,3,0,2,202,115,0,7
8,11,14,0,93995050,15,0,0,308,109,0,8
9,3,15,0,93995020,9,0,7,128,162,0,9


The next cell executes the full sort and writes a Parquet directory. It uses one worker process, one active task thread, a 2 GiB worker budget, and disk spilling. Leave RAM for the notebook and OS, and substantial free disk space for shuffle files and output. These are safeguards, not a strict memory guarantee: skewed/repeated timestamps can create oversized partitions. If the worker repeatedly restarts, stop and use an external merge sort or smaller, explicit time windows.

Equal timestamps have no guaranteed tie order. All event columns stay together. Reopen the output lazily for subsequent experiments.


In [ ]:
from pathlib import Path

output = Path("events_sorted_by_toa")
spill_directory = Path(".dask-spill")

sorted_events = sort_to_parquet(
    events, manifest, output, spill_directory,
    worker_memory="2GiB",
    output_partitions=events.npartitions,
)


2026-10-02 13:05:05,251 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle b6566e567597230e11c01ff5ba84c0ab initialized by task ('shuffle-transfer-b6566e567597230e11c01ff5ba84c0ab', 749) executed on worker tcp://127.0.0.1:51203
2026-10-02 13:05:49,935 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle b6566e567597230e11c01ff5ba84c0ab deactivated due to stimulus 'task-finished-1790960749.93418'
2026-10-02 13:05:50,269 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "/Users/7c2/Documents/GitHub/Kronos/.venv/lib/python3.14/site-packages/distributed/comm/tcp.py", line 226, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/7c2/Document

In [31]:
# In a later session, no H5 handles are needed.
sorted_events = dd.read_parquet("events_sorted_by_toa")
sorted_events.head(10, npartitions=1)
# Select a time window lazily: sorted_events.loc[start_toa:stop_toa]


,ftoa_fall,ftoa_rise,pileup,tot,uftoa_start,uftoa_stop,x,y,file_id,event_id
toa,,,,,,,,,,
93953928,3,5,0,11,0,0,200,255,5,93
93953928,1,5,0,11,0,0,200,254,5,74
93953929,7,0,0,11,0,3,120,300,21,56
93953930,4,1,0,2,0,6,245,177,5,17
93953930,12,15,0,8,3,5,231,249,5,97
93953930,6,6,0,23,0,7,246,177,5,40
93953930,6,12,0,13,0,5,242,207,5,42
93953930,14,12,0,8,0,4,242,208,5,50
93953930,0,7,0,25,0,3,173,228,5,38


In [32]:
ymin = sorted_events.y.min().compute()
ymax = sorted_events.y.max().compute()
xmin = sorted_events.x.min().compute()
xmax = sorted_events.x.max().compute()